# Portal Belajar SMA NU Kaplongan — Server Database Google Colab (SQLite + Google Drive)

Notebook ini menjadikan **Google Colab + Google Drive** sebagai server database SQLite terpusat untuk seluruh siswa, guru, dan admin.

### Cara Menjalankan:
1. Jalankan **Sel 1** untuk memasang Google Drive (agar data tersimpan permanen di Drive Anda) & menginstal library FastAPI + Cloudflared.
2. Jalankan **Sel 2** untuk mengatur password Guru/Admin dan menjalankan server database SQLite + Tunnel HTTPS publik.
3. Salin link **`https://portalbelajar-kaplongan.pages.dev/?colab=...`** yang muncul di output, lalu buka di browser atau bagikan ke siswa agar semua otomatis tersinkron!

In [ ]:
# @title 1. Mount Google Drive & Install Dependencies
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ Google Drive terpasang — database akan tersimpan permanen di MyDrive/portalbelajar_db/')
except Exception as e:
    print('Mode non-Colab / lokal:', e)

!pip install -q fastapi uvicorn python-multipart
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print('✓ Dependensi siap!')

In [ ]:
# @title 2. Jalankan Database Server & Tunnel Publik
import os, re, subprocess, threading, time
import urllib.request

# Ubah password akses guru dan admin sesuai kebutuhan:
os.environ['GURU_UPLOAD_PASSPHRASE'] = 'guru-kaplongan'   # @param {type:"string"}
os.environ['ADMIN_PASSWORD'] = 'admin-kaplongan'          # @param {type:"string"}
FRONTEND_URL = 'https://portalbelajar-kaplongan.pages.dev' # @param {type:"string"}
# Opsional (Rekomendasi Permanen): isi Token Cloudflare Named Tunnel agar URL tidak pernah berubah.
# Jika kosong, otomatis memakai Quick Tunnel (*.trycloudflare.com).
CF_TUNNEL_TOKEN = ''                                       # @param {type:"string"}
PERMANENT_TUNNEL_URL = ''                                  # @param {type:"string"}

# Pastikan file colab_server.py tersedia (jika belum ada, unduh atau gunakan yang sudah diunggah)
if not os.path.exists('colab_server.py'):
    raise FileNotFoundError('Upload file colab_server.py ke Google Colab atau jalankan dari folder proyek colab/')

# Jalankan server FastAPI di port 8000
api_proc = subprocess.Popen(['uvicorn', 'colab_server:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(2)

if CF_TUNNEL_TOKEN.strip():
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', CF_TUNNEL_TOKEN.strip()],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    print('\n' + '=' * 65)
    print('🟢 DATABASE GOOGLE COLAB AKTIF (NAMED TUNNEL PERMANEN)!')
    if PERMANENT_TUNNEL_URL.strip():
        print('URL Permanen        :', PERMANENT_TUNNEL_URL.strip())
        print('Link Portal Otomatis:', f'{FRONTEND_URL}/?colab={PERMANENT_TUNNEL_URL.strip()}')
    print('=' * 65 + '\n')
else:
    # Jalankan Cloudflare Quick Tunnel
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    public_url = None
    for line in tunnel_proc.stdout:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            print('\n' + '=' * 65)
            print('🟢 DATABASE GOOGLE COLAB AKTIF!')
            print('URL Database Colab :', public_url)
            print('Link Portal Otomatis:', f'{FRONTEND_URL}/?colab={public_url}')
            print('=' * 65 + '\n')
            break

# Pertahankan sel tetap berjalan
try:
    api_proc.wait()
except KeyboardInterrupt:
    api_proc.terminate()
    tunnel_proc.terminate()
    print('Server dihentikan.')